# VietMed-NER — PhoBERT vs ViHealthBERT (Softmax, full fine-tune) + Gradio

Reuse the PhoBERT Softmax checkpoint, full fine-tune ViHealthBERT with the **same config**,
compare entity-level F1 + error analysis, then a Gradio demo with an encoder picker.
Spec: `docs/superpowers/specs/2026-06-30-phobert-vs-vihealthbert-softmax-gradio-design.md`

In [ ]:
!pip -q install "transformers==4.44.2" "datasets==2.21.0" seqeval pytorch-crf py_vncorenlp pandas gradio
# §0 Setup
import os, random, numpy as np, torch
def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(42)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR   = '/content/drive/MyDrive/vietmed_ner'
CKPT_DIR    = f'{DRIVE_DIR}/checkpoints'
RESULTS_DIR = f'{DRIVE_DIR}/results'
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
# §1 Config — identical training config for both encoders (only the encoder differs)
CFG_PHO = dict(encoder="vinai/phobert-base",                head="softmax",
               segment=True, max_len=256, batch_size=16,
               ckpt=f'{CKPT_DIR}/01_phobert_softmax.pt.best')      # REUSE (no retrain)
CFG_VIH = dict(encoder="demdecuong/vihealthbert-base-word", head="softmax",
               segment=True, max_len=256, batch_size=16,
               ckpt=f'{CKPT_DIR}/02_vihealthbert_softmax.pt')      # TRAIN + save here
LR, EPOCHS, PATIENCE = 2e-5, 30, 3

In [ ]:
# §2 Data
from datasets import load_dataset
raw = load_dataset("leduckhai/VietMed-NER")
for c in ("audio", "duration"):
    if c in raw["train"].column_names:
        raw = raw.remove_columns(c)
TAG_COL  = "labels" if "labels" in raw["train"].column_names else "tags"
WORD_COL = "words"
def get_words_labels(split):
    ds = raw[split]; return list(ds[WORD_COL]), list(ds[TAG_COL])
_all_tags = set(t for split in raw for row in raw[split][TAG_COL] for t in row)
LABEL_LIST = ["O"] + sorted(t for t in _all_tags if t != "O")
label2id = {t:i for i,t in enumerate(LABEL_LIST)}
id2label = {i:t for t,i in label2id.items()}
print(len(LABEL_LIST), "labels; splits:", {k: len(raw[k]) for k in raw})

In [ ]:
# §3 Preprocess (reused from nb05)
def realign_tags(syllables, tags, word_groups):
    words, new_tags = [], []
    for grp in word_groups:
        words.append("_".join(syllables[i] for i in grp))
        head = tags[grp[0]]
        if head.startswith("I-"): head = "B-" + head[2:]
        new_tags.append(head)
    return words, new_tags

import py_vncorenlp
_SEG = None
def get_segmenter():
    global _SEG
    if _SEG is None:
        os.makedirs(f'{DRIVE_DIR}/vncorenlp', exist_ok=True)
        py_vncorenlp.download_model(save_dir=f'{DRIVE_DIR}/vncorenlp')
        _SEG = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=f'{DRIVE_DIR}/vncorenlp')
    return _SEG

def segment_to_groups(syllables):
    seg = get_segmenter()
    word_units = " ".join(seg.word_segment(" ".join(syllables))).split()
    groups, cur = [], 0
    for wu in word_units:
        n = wu.count("_") + 1
        groups.append(list(range(cur, cur + n))); cur += n
    if cur != len(syllables):
        return [[i] for i in range(len(syllables))]
    return groups

In [ ]:
# §3b Dual-path tokenization — fast (PhoBERT, word_ids) OR slow (ViHealthBERT, manual)
class _Encoded:
    def __init__(self, ids, mask, labels, wids):
        self._d = {"input_ids": ids, "attention_mask": mask, "labels": labels}; self._wid = wids
    def __getitem__(self, k): return self._d[k]
    def word_ids(self, batch_index=0): return self._wid[batch_index]

def encode_words(words, label_ids, tokenizer, max_len):
    cls, sep = tokenizer.cls_token_id, tokenizer.sep_token_id
    pad, unk = tokenizer.pad_token_id, tokenizer.unk_token_id
    ids, labs, wids = [cls], [-100], [None]
    for wi, (w, lid) in enumerate(zip(words, label_ids)):
        sub = tokenizer.encode(w, add_special_tokens=False) or [unk]
        for k, s in enumerate(sub):
            ids.append(s); labs.append(lid if k == 0 else -100); wids.append(wi)
    ids, labs, wids = ids[:max_len-1], labs[:max_len-1], wids[:max_len-1]
    ids.append(sep); labs.append(-100); wids.append(None)
    n = len(ids); mask = [1]*n + [0]*(max_len-n)
    ids += [pad]*(max_len-n); labs += [-100]*(max_len-n); wids += [None]*(max_len-n)
    return ids, mask, labs, wids

def align_labels(tok, word_label_ids):           # fast-path alignment via word_ids()
    aligned = []
    for i, labels in enumerate(word_label_ids):
        prev, ids = None, []
        for wid in tok.word_ids(batch_index=i):
            ids.append(-100 if (wid is None or wid == prev) else labels[wid]); prev = wid
        aligned.append(ids)
    tok["labels"] = aligned
    return tok

def _prep_words(split, cfg):
    words_col, tags_col = get_words_labels(split)
    enc_words, enc_lids = [], []
    for syl, tags in zip(words_col, tags_col):
        w, t = (realign_tags(syl, tags, segment_to_groups(syl)) if cfg["segment"] else (syl, tags))
        enc_words.append(w); enc_lids.append([label2id[x] for x in t])
    return enc_words, enc_lids

def build_encoded(split, cfg, tokenizer):
    enc_words, enc_lids = _prep_words(split, cfg)
    if tokenizer.is_fast:                          # PhoBERT — identical to nb05
        tok = tokenizer(enc_words, is_split_into_words=True, truncation=True,
                        max_length=cfg["max_len"], padding="max_length")
        return align_labels(tok, enc_lids)
    ids, mask, labs, wids = [], [], [], []         # ViHealthBERT — manual
    for w, l in zip(enc_words, enc_lids):
        a, b, c, d = encode_words(w, l, tokenizer, cfg["max_len"])
        ids.append(a); mask.append(b); labs.append(c); wids.append(d)
    return _Encoded(ids, mask, labs, wids)

In [ ]:
# §T tests — pure functions, CPU-only
def _test_realign():
    w,t = realign_tags(["bệnh","nhân","bị"], ["B-AGE","I-AGE","O"], [[0,1],[2]])
    assert w==["bệnh_nhân","bị"] and t==["B-AGE","O"]
    assert realign_tags(["a","b"], ["O","I-AGE"], [[0],[1]])[1]==["O","B-AGE"]
    print("realign_tags OK")
class _FakeTok:
    is_fast=False; cls_token_id,sep_token_id,pad_token_id,unk_token_id=1,2,3,0
    def encode(self,w,add_special_tokens=False): return {"ab":[10,11],"c":[12]}[w]
def _test_encode():
    ids,mask,labs,wids = encode_words(["ab","c"],[5,7],_FakeTok(),7)
    assert ids==[1,10,11,12,2,3,3] and labs==[-100,5,-100,7,-100,-100,-100]
    assert wids==[None,0,0,1,None,None,None] and mask==[1,1,1,1,1,0,0]
    print("encode_words OK")
_test_realign(); _test_encode()
print("ALL §T (part 1) PASSED")

In [ ]:
# §4 Verify (GATE) — roundtrip both tokenizer paths before any train/eval
from transformers import AutoTokenizer

def load_tokenizer(cfg):
    if cfg["encoder"] == "vinai/phobert-base":
        return AutoTokenizer.from_pretrained(cfg["encoder"], use_fast=True, add_prefix_space=True)
    return AutoTokenizer.from_pretrained(cfg["encoder"], use_fast=False)   # ViHealthBERT slow

def verify_roundtrip(cfg, tokenizer, n=5):
    tok = build_encoded("train", cfg, tokenizer)
    words_col, tags_col = get_words_labels("train"); ok = 0
    for i in range(n):
        wids, labs = tok.word_ids(batch_index=i), tok["labels"][i]
        rec, seen = [], set()
        for wid, lab in zip(wids, labs):
            if wid is None or wid in seen: continue
            seen.add(wid); rec.append(id2label[lab])
        syl, tags = words_col[i], tags_col[i]
        exp = realign_tags(syl, tags, segment_to_groups(syl))[1] if cfg["segment"] else tags
        assert rec == exp[:len(rec)], f"mismatch {i}: {rec} vs {exp[:len(rec)]}"
        ok += 1
    print(f"verify_roundtrip PASSED ({cfg['encoder']}) {ok}/{n}")

for _cfg in (CFG_PHO, CFG_VIH):
    verify_roundtrip(_cfg, load_tokenizer(_cfg))

In [ ]:
# §5 Model builder (Softmax head)
from transformers import AutoModelForTokenClassification

def build_softmax_model(cfg, tokenizer):
    model = AutoModelForTokenClassification.from_pretrained(
        cfg["encoder"], num_labels=len(LABEL_LIST), id2label=id2label, label2id=label2id)
    model.roberta.resize_token_embeddings(len(tokenizer))
    return model

In [ ]:
# Eval helpers (reused from nb05)
from seqeval.metrics import f1_score, classification_report
from torch.utils.data import DataLoader, TensorDataset

def evaluate_tags(true_tags, pred_tags):
    return {"micro_f1": f1_score(true_tags, pred_tags, average="micro"),
            "macro_f1": f1_score(true_tags, pred_tags, average="macro"),
            "report":   classification_report(true_tags, pred_tags, digits=4)}

def decode_predictions(pred, label_ids, head):
    T, P = [], []
    for i, gold in enumerate(label_ids):
        p_seq = pred[i]; t_row, p_row, j = [], [], 0
        for k, g in enumerate(gold):
            if g == -100: continue
            t_row.append(id2label[g])
            p_row.append(id2label[p_seq[k]] if head == "softmax" else id2label[p_seq[j]]); j += 1
        T.append(t_row); P.append(p_row)
    return T, P

def _loader(cfg, tokenizer, split):
    tok = build_encoded(split, cfg, tokenizer)
    ds = TensorDataset(torch.tensor(tok["input_ids"]), torch.tensor(tok["attention_mask"]),
                       torch.tensor(tok["labels"]))
    return DataLoader(ds, batch_size=cfg["batch_size"]), tok

def eval_softmax(cfg, tokenizer, model, split="test"):
    dl, _ = _loader(cfg, tokenizer, split); model.eval(); T, P = [], []
    with torch.no_grad():
        for ids, mask, labs in dl:
            pred = model(ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits.argmax(-1).cpu().tolist()
            t, p = decode_predictions(pred, labs.tolist(), "softmax"); T += t; P += p
    return evaluate_tags(T, P)

In [ ]:
# §6 PhoBERT — load existing checkpoint, evaluate on test (NO training)
tok_pho   = load_tokenizer(CFG_PHO)
model_pho = build_softmax_model(CFG_PHO, tok_pho).to(DEVICE)
state = torch.load(CFG_PHO["ckpt"], map_location=DEVICE)
model_pho.load_state_dict(state)
m_pho = eval_softmax(CFG_PHO, tok_pho, model_pho, "test")
print("PhoBERT  test micro_f1 =", round(m_pho["micro_f1"], 4), " macro_f1 =", round(m_pho["macro_f1"], 4))
assert m_pho["micro_f1"] > 0.5, "PhoBERT load/preproc mismatch (expected ~0.8+)"

In [ ]:
# §7 ViHealthBERT — full fine-tune Softmax (same config), save checkpoint, eval
from transformers import get_linear_schedule_with_warmup

def train_softmax(cfg, tokenizer):
    set_seed(42)
    dl_tr, _ = _loader_shuffle(cfg, tokenizer, "train")
    model = build_softmax_model(cfg, tokenizer).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = len(dl_tr) * EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(0.1*total), total)
    scaler = torch.cuda.amp.GradScaler()
    best, bad = -1.0, 0
    for ep in range(EPOCHS):
        model.train()
        for ids, mask, labs in dl_tr:
            ids, mask, labs = ids.to(DEVICE), mask.to(DEVICE), labs.to(DEVICE)
            with torch.cuda.amp.autocast():
                loss = model(ids, attention_mask=mask, labels=labs).loss
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        mv = eval_softmax(cfg, tokenizer, model, "validation")
        print(f"ep{ep} val micro_f1={mv['micro_f1']:.4f}")
        torch.save(model.state_dict(), cfg["ckpt"])
        if mv["micro_f1"] > best:
            best, bad = mv["micro_f1"], 0; torch.save(model.state_dict(), cfg["ckpt"]+".best")
        else:
            bad += 1
            if bad >= PATIENCE: print("early stop"); break
    model.load_state_dict(torch.load(cfg["ckpt"]+".best", map_location=DEVICE))
    return model, eval_softmax(cfg, tokenizer, model, "test")

def _loader_shuffle(cfg, tokenizer, split):
    tok = build_encoded(split, cfg, tokenizer)
    ds = TensorDataset(torch.tensor(tok["input_ids"]), torch.tensor(tok["attention_mask"]),
                       torch.tensor(tok["labels"]))
    return DataLoader(ds, batch_size=cfg["batch_size"], shuffle=True), tok

tok_vih = load_tokenizer(CFG_VIH)
model_vih, m_vih = train_softmax(CFG_VIH, tok_vih)
print("ViHealthBERT test micro_f1 =", round(m_vih["micro_f1"], 4), " macro_f1 =", round(m_vih["macro_f1"], 4))
assert m_vih["micro_f1"] > 0.5, "ViHealthBERT training degenerate"